In [ ]:
"""
MMS TRANSITOIRE — vérification du modèle complet dans son cadre général

Système vérifié  :

  (1)  div( Kf grad h ) + div( Kf * buoy(C) * e_z ) = S_h(x,z,t)

  (2)  ne dC/dt + div( q C ) - div( D(q) grad C ) = S_C(x,z,t)

Solution manufacturée dépendante du temps :
  C_ex(x,z,t)  et h_ex(x,z,t) 

Les sources S_h et S_C (incluant ne dC_ex/dt) sont calculées par sympy.

Protocole de convergence : raffinement SIMULTANÉ dx et dt (dt proportionnel
à dx), erreur L2 mesurée au temps final T.
"""

import numpy as np
import sympy as sp
from fipy import (Grid2D, CellVariable, FaceVariable,
                  DiffusionTerm, UpwindConvectionTerm, TransientTerm)
from fipy.solvers.scipy import LinearLUSolver

# ------------------------------------------------------------------
# Paramètres physiques (identiques à la simulation)
# ------------------------------------------------------------------
Kf, ne           = 1e-4, 0.25
rho_f, rho_s     = 1000.0, 1025.0
C_mer            = 35.0
drho_dC          = (rho_s - rho_f) / C_mer
alpha_L, alpha_T = 1.0, 0.1
Dm               = 1e-9
L, H             = 100.0, 50.0

T_fin = 1.0e7          # temps final [s]  (~116 jours)
omega = 2*np.pi / (2*T_fin)   # une demi-période sur la simulation


# 1) Solutions manufacturées (x, z, t)
x, z, t = sp.symbols('x z t', real=True)

modul = 1 + sp.Rational(1, 2)*sp.sin(omega*t)          # modulation temporelle
h_ex = (sp.Float(0.02)*sp.sin(sp.pi*x/L)*sp.sin(sp.pi*z/H)
        - sp.Float(0.01)*z/H) * modul
C_ex = sp.Float(C_mer/2)*(1 + sp.Rational(4, 5)
                          * sp.sin(sp.pi*x/L)
                          * sp.cos(sp.pi*z/(2*H)) * modul)   # reste dans ]0,35[

buoy_ex = drho_dC*C_ex/rho_f
qx_ex = -Kf*sp.diff(h_ex, x)
qz_ex = -Kf*(sp.diff(h_ex, z) + buoy_ex)
qn_ex = sp.sqrt(qx_ex**2 + qz_ex**2)

Dxx_ex = alpha_L*qx_ex**2/qn_ex + alpha_T*qz_ex**2/qn_ex + ne*Dm
Dzz_ex = alpha_L*qz_ex**2/qn_ex + alpha_T*qx_ex**2/qn_ex + ne*Dm
Dxz_ex = (alpha_L - alpha_T)*qx_ex*qz_ex/qn_ex

# S_h : résidu de l'équation quasi-statique en h (à t donné)
S_h_ex = (sp.diff(Kf*sp.diff(h_ex, x), x)
          + sp.diff(Kf*sp.diff(h_ex, z), z)
          + sp.diff(Kf*buoy_ex, z))

# S_C : résidu du transport TRANSITOIRE — avec ne dC/dt
Fx = qx_ex*C_ex - (Dxx_ex*sp.diff(C_ex, x) + Dxz_ex*sp.diff(C_ex, z))
Fz = qz_ex*C_ex - (Dxz_ex*sp.diff(C_ex, x) + Dzz_ex*sp.diff(C_ex, z))
S_C_ex = ne*sp.diff(C_ex, t) + sp.diff(Fx, x) + sp.diff(Fz, z)

f = lambda e: sp.lambdify((x, z, t), e, 'numpy')
h_f, C_f     = f(h_ex), f(C_ex)
S_h_f, S_C_f = f(S_h_ex), f(S_C_ex)

# 2) Résolution transitoire sur une grille N, avec M pas de temps
def solve_transient(N, M, n_picard=6):
    Nx, Nz = N, N // 2
    dx = L / Nx
    dt = T_fin / M
    mesh = Grid2D(dx=dx, dy=dx, nx=Nx, ny=Nz)
    solver = LinearLUSolver(tolerance=1e-13, iterations=2000)

    xc = np.array(mesh.cellCenters[0]); zc = np.array(mesh.cellCenters[1])
    xf = np.array(mesh.faceCenters[0]); zf = np.array(mesh.faceCenters[1])
    nF = mesh.numberOfFaces

    # les contraintes Dirichlet doivent SUIVRE le temps : on utilise des
    # FaceVariable dont on met à jour la valeur à chaque pas
    hBC = FaceVariable(mesh=mesh, value=h_f(xf, zf, 0.0))
    CBC = FaceVariable(mesh=mesh, value=C_f(xf, zf, 0.0))
    h = CellVariable(mesh=mesh, value=h_f(xc, zc, 0.0))
    C = CellVariable(mesh=mesh, value=C_f(xc, zc, 0.0), hasOld=True)
    h.constrain(hBC, mesh.exteriorFaces)
    C.constrain(CBC, mesh.exteriorFaces)

    Kf_face  = FaceVariable(mesh=mesh, value=Kf)
    Kf_vals  = np.full(nF, Kf)
    rho_cell = CellVariable(mesh=mesh, value=rho_f)
    rho_face = rho_cell.arithmeticFaceValue

    b_var  = CellVariable(mesh=mesh, value=0.0)
    S_C_c  = CellVariable(mesh=mesh, value=0.0)
    D_face = FaceVariable(mesh=mesh, rank=2)
    q_face = FaceVariable(mesh=mesh, rank=1)
    q_face.setValue(np.zeros((2, nF)))
    buoy_flux = FaceVariable(mesh=mesh, rank=1)

    eq_h = DiffusionTerm(coeff=Kf_face) == b_var
    eq_C = (TransientTerm(coeff=ne)
            + UpwindConvectionTerm(coeff=q_face)
            - DiffusionTerm(coeff=D_face)) == S_C_c

    res_cont = 0.0
    for n in range(M):
        tn1 = (n + 1) * dt                      # Euler implicite : tout à t^{n+1}
        hBC.setValue(h_f(xf, zf, tn1))
        CBC.setValue(C_f(xf, zf, tn1))
        S_h_now = S_h_f(xc, zc, tn1)
        S_C_c.setValue(S_C_f(xc, zc, tn1))
        C.updateOld()

        # couplage h <-> C au sein du pas (Picard court)
        for _ in range(n_picard):
            rho_cell.setValue(rho_f + drho_dC*C.value)
            buoy = (np.array(rho_face) - rho_f)/rho_f
            bf = np.zeros((2, nF)); bf[1] = Kf_vals*buoy
            buoy_flux.setValue(bf)
            b_var.setValue(-np.array(buoy_flux.divergence) + S_h_now)
            eq_h.solve(var=h, solver=solver)

            h_fg = h.faceGrad.value
            qx = -Kf_vals*h_fg[0]
            qz = -Kf_vals*(h_fg[1] + buoy)
            q_face.setValue(np.array([qx, qz]))

            qn = np.sqrt(qx**2 + qz**2) + 1e-300
            Da = np.zeros((2, 2, nF))
            Da[0, 0] = alpha_L*qx**2/qn + alpha_T*qz**2/qn + ne*Dm
            Da[1, 1] = alpha_L*qz**2/qn + alpha_T*qx**2/qn + ne*Dm
            Da[0, 1] = Da[1, 0] = (alpha_L - alpha_T)*qx*qz/qn
            D_face.setValue(Da)

            eq_C.solve(var=C, dt=dt, solver=solver)

        tmp = FaceVariable(mesh=mesh, rank=1, value=np.array([qx, qz]))
        res_cont = max(res_cont,
                       np.abs(np.array(tmp.divergence) + S_h_now).max())

    V = dx*dx
    err_h = np.sqrt(np.sum((h.value - h_f(xc, zc, T_fin))**2) * V)
    err_C = np.sqrt(np.sum((C.value - C_f(xc, zc, T_fin))**2) * V)
    return dx, dt, err_h, err_C, res_cont

# ------------------------------------------------------------------
# 3) Convergence : raffinement simultané dx, dt  (dt proportionnel à dx)
# ------------------------------------------------------------------
print("MMS TRANSITOIRE — raffinement simultané espace/temps (dt ~ dx)")
print(f"{'N':>5} {'M':>5} {'dx [m]':>8} {'dt [s]':>9} "
      f"{'||h-h_ex||L2':>13} {'ordre':>6} {'||C-C_ex||L2':>13} {'ordre':>6} "
      f"{'|div q + S_h|':>13}")

res = []
for N, M in [(16, 10), (32, 20), (64, 40), (128, 80), (256, 160)]:
    dx, dt, eh, eC, rc = solve_transient(N, M)
    o_h = np.log2(res[-1][0]/eh) if res else float('nan')
    o_C = np.log2(res[-1][1]/eC) if res else float('nan')
    res.append((eh, eC))
    print(f"{N:>5} {M:>5} {dx:>8.3f} {dt:>9.1e} "
          f"{eh:>13.4e} {o_h:>6.2f} {eC:>13.4e} {o_C:>6.2f} {rc:>13.2e}")

print("\nAttendu : ordre global ~1 (Euler implicite O(dt) + upwind O(dx)),")
print("résidu de continuité discret au zéro machine à chaque pas.")